# Global Missing Migrants Dataset: Data Manipulation and Statistical Analysis

**Purpose:** demonstrate foundational data structures, data cleaning, statistical methods, visualization, and responsible interpretation using Python.

**Tools:** Pandas, NumPy, Matplotlib, Seaborn.

**Important interpretation note:** this dataset represents documented incidents. It may be incomplete and should not be treated as a complete count of all deaths and disappearances or as a direct measure of population-level risk.

## Research questions
1. What is the structure and quality of the dataset?
2. How complete are key fields, and are exact duplicate rows present?
3. How do documented records vary over time and across categories such as region, route, or cause?
4. What do descriptive statistics and correlations tell us about numeric fields?
5. What limitations should decision-makers consider?

## How to use this notebook
Place the source CSV in the same folder as this notebook. Run the notebook from top to bottom. If the CSV has a different filename, update `FILE_PATH` in the setup cell. Column names can vary between dataset releases; the notebook detects common field names and prints the fields it selects.

## 1. Import libraries and locate the dataset

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 100)

# Edit this path if your CSV is stored elsewhere.
candidate_paths = [
    Path("Global Missing Migrants Dataset.csv"),
    Path("Global_Missing_Migrants_Dataset.csv"),
    Path("global_missing_migrants.csv"),
    Path("/mnt/data/Global Missing Migrants Dataset.csv"),
    Path("/mnt/data/Global_Missing_Migrants_Dataset.csv"),
]
FILE_PATH = next((p for p in candidate_paths if p.exists()), None)

if FILE_PATH is None:
    matches = [
        p for p in Path.cwd().glob("*.csv")
        if "missing" in p.name.lower() and "migrant" in p.name.lower()
    ]
    if len(matches) == 1:
        FILE_PATH = matches[0]

if FILE_PATH is None:
    raise FileNotFoundError(
        "CSV not found. Put the dataset CSV next to this notebook, "
        "or edit FILE_PATH to the correct filename/path."
    )

print("Dataset:", FILE_PATH.resolve())

## 2. Load and inspect the data

A Pandas DataFrame stores tabular data. A Series represents one column. First check the dimensions, column names, sample records, and data types.

In [ ]:
df_raw = pd.read_csv(FILE_PATH, low_memory=False)
df = df_raw.copy()

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())
display(pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "non_missing": df.notna().sum().values,
    "missing": df.isna().sum().values,
    "missing_percent": (df.isna().mean().values * 100).round(2),
}))

## 3. Standardize column names and blank values

The original data are preserved in `df_raw`. In the working copy, column names are made easier to use in code. Empty text fields become missing values; meaningful categories such as `Unknown` are retained.

In [ ]:
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
    .str.strip("_")
)

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)
    df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

print("Standardized columns:")
print(df.columns.tolist())

## 4. Data quality: missing values and exact duplicates

Missing values are not the same as zero. Exact duplicate rows are counted, but not automatically removed: two rows that look identical do not always prove that the same real-world incident was duplicated. Review the source documentation before deciding whether to remove them.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "non_missing_count": df.notna().sum(),
}).sort_values("missing_percent", ascending=False)

display(missing_summary)

duplicate_count = int(df.duplicated().sum())
print(f"Exact duplicate rows detected: {duplicate_count:,}")

REMOVE_EXACT_DUPLICATES = False  # Set True only after reviewing the source.
df_clean = df.drop_duplicates().copy() if REMOVE_EXACT_DUPLICATES else df.copy()

print(f"Rows before optional duplicate removal: {len(df):,}")
print(f"Rows used for analysis: {len(df_clean):,}")
print(f"Exact duplicate removal enabled: {REMOVE_EXACT_DUPLICATES}")

## 5. Identify date, year, category, and count fields

Column names vary by release. The detection below prioritizes common exact names and then checks keywords. Review the selected fields printed below before interpreting any chart.

In [ ]:
def first_matching_column(columns, preferred_names=(), keywords=()):
    for name in preferred_names:
        if name in columns:
            return name
    for col in columns:
        if any(keyword in col.lower() for keyword in keywords):
            return col
    return None

columns = df_clean.columns.tolist()

DATE_COL = first_matching_column(
    columns,
    preferred_names=("incident_date", "date_of_incident", "reported_date"),
    keywords=("date",)
)
YEAR_COL = first_matching_column(
    columns,
    preferred_names=("incident_year", "year"),
    keywords=("year",)
)
MONTH_COL = first_matching_column(
    columns,
    preferred_names=("incident_month", "reported_month", "month"),
    keywords=("month",)
)
REGION_COL = first_matching_column(
    columns,
    preferred_names=("region_of_incident", "region"),
    keywords=("region", "subregion")
)
ROUTE_COL = first_matching_column(
    columns,
    preferred_names=("migration_route", "route"),
    keywords=("route",)
)
CAUSE_COL = first_matching_column(
    columns,
    preferred_names=("cause_of_death", "cause"),
    keywords=("cause_of_death", "cause")
)
COUNTRY_COL = first_matching_column(
    columns,
    preferred_names=("country_of_incident", "country_of_origin", "country"),
    keywords=("country",)
)
COUNT_COL = first_matching_column(
    columns,
    preferred_names=(
        "total_number_of_dead_and_missing",
        "number_of_dead_and_missing",
        "number_dead",
        "total_dead_and_missing",
    ),
    keywords=("total_number_of_dead_and_missing", "number_of_dead_and_missing", "number_dead")
)

selected_fields = {
    "date": DATE_COL,
    "year": YEAR_COL,
    "month": MONTH_COL,
    "region": REGION_COL,
    "route": ROUTE_COL,
    "cause": CAUSE_COL,
    "country": COUNTRY_COL,
    "count_measure": COUNT_COL,
}
display(pd.DataFrame({"selected_column": selected_fields}))

## 6. Convert date and numeric fields safely

A year value such as `2014` should not be passed directly to `to_datetime()` as if it were a timestamp. Full date fields and year fields are handled separately.

In [ ]:
if DATE_COL:
    parsed_dates = pd.to_datetime(df_clean[DATE_COL], errors="coerce")
    existing = df_clean[DATE_COL].notna().sum()
    parse_rate = parsed_dates.notna().sum() / existing if existing else 0
    print(f"Date parsing success rate for {DATE_COL}: {parse_rate:.1%}")
    if existing and parse_rate >= 0.70:
        df_clean[DATE_COL] = parsed_dates

if YEAR_COL:
    df_clean[YEAR_COL] = pd.to_numeric(df_clean[YEAR_COL], errors="coerce")
    valid_years = df_clean[YEAR_COL].between(1900, 2100)
    print(f"Valid-looking year values in {YEAR_COL}: {valid_years.sum():,} / {df_clean[YEAR_COL].notna().sum():,}")

if COUNT_COL:
    df_clean[COUNT_COL] = pd.to_numeric(df_clean[COUNT_COL], errors="coerce")

display(df_clean.dtypes.to_frame("dtype"))

## 7. Descriptive statistics

Mean, median, standard deviation, variance, and percentiles summarize numeric variables. The mean can be influenced by a small number of very large incidents, so compare it with the median and quartiles. These are descriptive statistics, not causal estimates.

In [ ]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()

if numeric_cols:
    display(df_clean[numeric_cols].describe().T)
else:
    print("No numeric columns were detected.")

## 8. Distributions: histograms and boxplots

Histograms show the shape of a variable's distribution. Boxplots help inspect spread and possible extreme values. An extreme value should be investigated, not automatically removed.

In [ ]:
plot_cols = [c for c in numeric_cols if df_clean[c].notna().sum() > 0][:6]

for col in plot_cols:
    values = df_clean[col].dropna()
    if values.empty:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(values, bins="auto", edgecolor="black")
    axes[0].set_title(f"Histogram: {col}")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Frequency")

    axes[1].boxplot(values, vert=False)
    axes[1].set_title(f"Boxplot: {col}")
    axes[1].set_xlabel(col)
    plt.tight_layout()
    plt.show()

## 9. Documented records over time

This chart counts rows by year. It does **not** directly measure the true number of migration-related deaths and disappearances. Trends may reflect real-world changes as well as reporting coverage, classification, and documentation.

In [ ]:
annual_records = None

if DATE_COL and pd.api.types.is_datetime64_any_dtype(df_clean[DATE_COL]):
    years = df_clean[DATE_COL].dropna().dt.year
    if not years.empty:
        annual_records = years.value_counts().sort_index()
elif YEAR_COL:
    years = pd.to_numeric(df_clean[YEAR_COL], errors="coerce")
    years = years[years.between(1900, 2100)].astype(int)
    if not years.empty:
        annual_records = years.value_counts().sort_index()

if annual_records is not None and not annual_records.empty:
    display(annual_records.rename("record_count").to_frame())
    annual_records.plot(marker="o")
    plt.title("Documented records by year")
    plt.xlabel("Year")
    plt.ylabel("Number of records")
    plt.tight_layout()
    plt.show()
else:
    print("No usable full-date or year field was detected. Check the selected fields in Section 5.")

## 10. Category analysis

The following charts show the most frequent recorded categories. A category with more records does not necessarily have greater underlying risk: route volume, reporting practices, and data coverage may differ.

In [ ]:
for col in [REGION_COL, ROUTE_COL, CAUSE_COL, COUNTRY_COL]:
    if col:
        counts = df_clean[col].fillna("Missing").value_counts().head(15)
        print(f"Top categories for {col}")
        display(counts.rename("record_count").to_frame())

        counts.sort_values().plot(kind="barh")
        plt.title(f"Top recorded categories: {col}")
        plt.xlabel("Number of records")
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()

## 11. Grouped summaries for numeric measures

`groupby()` summarizes data by category. The notebook reports record counts and, when a numeric count measure is available, selected descriptive statistics. Interpret each measure using its data dictionary definition.

In [ ]:
group_cols = [c for c in [REGION_COL, ROUTE_COL, CAUSE_COL] if c]

for group_col in group_cols:
    print(f"Record counts by {group_col}")
    display(
        df_clean.groupby(group_col, dropna=False)
        .size().sort_values(ascending=False)
        .head(20).rename("record_count").to_frame()
    )

    if COUNT_COL:
        print(f"Summary of {COUNT_COL} by {group_col}")
        display(
            df_clean.groupby(group_col, dropna=False)[COUNT_COL]
            .agg(["count", "mean", "median", "sum", "max"])
            .sort_values("count", ascending=False)
            .head(20)
        )

## 12. Correlation analysis

Pearson correlation measures linear association between numeric variables. It does not prove causation. Some incident-count fields may be mathematically related, so inspect variable definitions before interpreting high correlations.

In [ ]:
count_keywords = ("number", "total", "dead", "missing", "surviv", "female", "male", "child")
corr_cols = [
    c for c in df_clean.select_dtypes(include=np.number).columns
    if any(k in c.lower() for k in count_keywords)
    and "year" not in c.lower()
    and not c.lower().endswith("_id")
]
corr_cols = [
    c for c in corr_cols
    if df_clean[c].notna().sum() >= 3 and df_clean[c].nunique(dropna=True) > 1
]

if len(corr_cols) >= 2:
    corr = df_clean[corr_cols].corr(method="pearson", min_periods=3)
    display(corr.round(2))
    plt.figure(figsize=(max(8, len(corr_cols) * 0.8), 6))
    sns.heatmap(corr, cmap="vlag", center=0, vmin=-1, vmax=1, annot=True, fmt=".2f")
    plt.title("Correlation among selected numeric count variables")
    plt.tight_layout()
    plt.show()
else:
    print("Fewer than two suitable numeric count variables were detected.")

## 13. Geographic exploration, if coordinates are available

This section supports separate latitude/longitude columns or a combined coordinate field containing `latitude, longitude`. Coordinates are validated against geographic ranges. The plot is a view of recorded coordinates, not a population-risk map.

In [ ]:
lat_candidates = [c for c in df_clean.columns if c in ("lat", "latitude") or c.endswith("_latitude")]
lon_candidates = [c for c in df_clean.columns if c in ("lon", "lng", "longitude") or c.endswith("_longitude")]
coord_candidates = [c for c in df_clean.columns if "coordinate" in c.lower()]

geo = None
if lat_candidates and lon_candidates:
    geo = pd.DataFrame({
        "latitude": pd.to_numeric(df_clean[lat_candidates[0]], errors="coerce"),
        "longitude": pd.to_numeric(df_clean[lon_candidates[0]], errors="coerce"),
    })
elif coord_candidates:
    parts = df_clean[coord_candidates[0]].astype("string").str.split(",", n=1, expand=True)
    if parts.shape[1] == 2:
        geo = pd.DataFrame({
            "latitude": pd.to_numeric(parts[0].str.strip(), errors="coerce"),
            "longitude": pd.to_numeric(parts[1].str.strip(), errors="coerce"),
        })

if geo is not None:
    geo = geo.dropna()
    geo = geo[geo["latitude"].between(-90, 90) & geo["longitude"].between(-180, 180)]
    print(f"Records with valid coordinate pairs: {len(geo):,}")
    if len(geo) > 10000:
        geo = geo.sample(10000, random_state=42)
        print("Showing a reproducible sample of 10,000 points.")
    if not geo.empty:
        plt.figure(figsize=(10, 6))
        plt.scatter(geo["longitude"], geo["latitude"], alpha=0.35, s=10)
        plt.title("Recorded incident coordinates")
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.tight_layout()
        plt.show()
    else:
        print("Coordinate fields exist, but no valid coordinate pairs remain.")
else:
    print("No separate latitude/longitude fields or combined coordinate field detected.")

## 14. Export analysis outputs

This exports the cleaned working dataset and missing-value summary to an `analysis_outputs` folder. Check the exported files before sharing them. The cleaning decisions above should be documented in any published analysis.

In [ ]:
OUTPUT_DIR = Path("analysis_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df_clean.to_csv(OUTPUT_DIR / "global_missing_migrants_cleaned.csv", index=False)
missing_summary.to_csv(OUTPUT_DIR / "missing_values_summary.csv")

if annual_records is not None and not annual_records.empty:
    annual_records.rename("record_count").to_csv(OUTPUT_DIR / "documented_records_by_year.csv")

print("Outputs saved to:", OUTPUT_DIR.resolve())
print("Files:", [p.name for p in OUTPUT_DIR.iterdir()])

## 15. Conclusions and responsible interpretation

Use the actual outputs generated above to write concise findings. Do not claim results that are not supported by the run.

When writing conclusions:
- State the number of records and columns analyzed.
- Identify important missing-data and duplicate issues.
- Describe observed trends or category patterns, naming the field counted.
- Distinguish the number of recorded incidents from the number of people reported dead or missing.
- Note that missingness, underreporting, inconsistent definitions, and reporting coverage can affect the findings.
- Avoid interpreting raw record counts as direct estimates of risk without appropriate population or exposure denominators.
- Correlation does not imply causation, and extreme values should be investigated before being excluded.

**Suggested interpretation:** This analysis describes patterns in documented incidents within the dataset. It does not establish the full number of deaths or disappearances, and differences between regions may reflect differences in exposure, reporting, and documentation.

## 16. Foundational concepts demonstrated

| Concept | Example in this notebook |
|---|---|
| Data structures | DataFrames, Series, column types |
| Data cleaning | Standardized names, blank values, missingness, duplicate review |
| Data transformation | Date parsing and numeric conversion |
| Descriptive statistics | Mean, median, standard deviation, percentiles |
| Grouped analysis | Counts and summaries by region, route, or cause |
| Correlation | Pearson correlation between suitable numeric variables |
| Visualization | Histograms, boxplots, bar charts, trends, heatmaps, coordinate scatterplot |
| Responsible analysis | Documentation of uncertainty, missingness, and reporting bias |

### Suggested portfolio description

**Global Missing Migrants Data Analysis:** A Python-based exploratory data analysis project demonstrating data cleaning, data structures, descriptive statistics, grouped analysis, correlation, visualization, and responsible interpretation of humanitarian incident data.